# Theoria v3 — conservative QLoRA for Qwen3-1.7B

This notebook makes a **small, eval-gated Theoria adapter**. It teaches identity, concise math style, core Lean 4 formalization, and short science answers while replaying the base model's own behavior to reduce forgetting.

It deliberately does **not** continue from the mikromini Q4 GGUF and does not reuse the old 9k-row mixed corpus.

## Runtime

1. In Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Run every cell in order. Keep the browser open during training.
3. When prompted, upload:
   - `data/finetune/seed_identity.json`
   - `data/finetune/sorry_fill.jsonl`
4. Expected time on a T4: roughly 45–90 minutes, including replay and evaluations.
5. Export only when the final gate prints `SHIP_CANDIDATE = True`.

The output is `theoria-v3-q4_k_m.gguf`, designed for llama.cpp on the ADTC 8 GB laptop profile.

In [ ]:
# 1) Install Unsloth, then pin deps to versions Unsloth actually supports.
# Do NOT `-U datasets` / `-U trl` — that pulls datasets 5.x and trl 1.x and breaks SFTTrainer.
# Restart only if Colab asks; then re-run this cell and continue from cell 2.
%pip install -q -U unsloth
%pip install -q \
  "datasets>=3.4.1,<4.4.0" "datasets!=4.0.*" "datasets!=4.1.0" \
  "trl>=0.18.2,<=0.24.0,!=0.19.0" \
  peft accelerate bitsandbytes sentencepiece "protobuf>=3.20.2,<6"

import datasets, trl, torch
print("datasets", datasets.__version__)
print("trl", trl.__version__)
assert torch.cuda.is_available(), "Select a T4 GPU runtime before continuing"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2) Upload only the hand-curated local sources. Remote math/science data is
# fetched below; the old train.jsonl is intentionally not accepted.
# Colab renames re-uploads as "seed_identity (1).json" — normalize names below.
from google.colab import files
import os, random, re, json, hashlib, glob, shutil

uploaded = files.upload()
print("Uploaded keys:", sorted(uploaded.keys()))

def resolve_upload(canonical: str) -> str:
    """Map Colab's 'name (1).ext' re-uploads back to canonical filenames."""
    if os.path.isfile(canonical):
        return canonical
    stem, ext = os.path.splitext(canonical)
    # Prefer the newest "stem (N).ext" from this or prior uploads.
    candidates = sorted(
        glob.glob(f"{stem} (*{ext}") + glob.glob(f"{stem}*{ext}"),
        key=lambda p: os.path.getmtime(p),
        reverse=True,
    )
    for path in candidates:
        base = os.path.basename(path)
        if base == canonical or base.startswith(stem + " (") or base.startswith(stem + "("):
            shutil.copyfile(path, canonical)
            print(f"Normalized {base!r} -> {canonical!r}")
            return canonical
    # Also accept if the upload dict used a weird key but wrote bytes under another name.
    for key in uploaded:
        if key == canonical or key.startswith(stem + " (") or key.startswith(stem + "("):
            with open(canonical, "wb") as f:
                f.write(uploaded[key])
            print(f"Wrote {key!r} -> {canonical!r}")
            return canonical
    raise FileNotFoundError(f"Need {canonical}. Got uploads: {sorted(uploaded.keys())}")

SEED_PATH = resolve_upload("seed_identity.json")
SORRY_PATH = resolve_upload("sorry_fill.jsonl")
assert os.path.isfile(SEED_PATH) and os.path.isfile(SORRY_PATH)

SEED = 42
BASE_MODEL = "unsloth/Qwen3-1.7B"
MAX_SEQ_LEN = 1024
REPLAY_N = 120
MATH_N = 260
SCIENCE_N = 60
LEARNING_RATE = 5e-5
LORA_R = 8

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
print("Configuration ready")
print("seed_identity.json bytes:", os.path.getsize(SEED_PATH))
print("sorry_fill.jsonl bytes:", os.path.getsize(SORRY_PATH))

In [ ]:
# 3) Load the instruct model in 4-bit and attach a deliberately low-capacity
# adapter. Fresh LoRA weights are zero, so this is still the base model.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Trainable adapter attached; base weights remain frozen")

In [ ]:
# 4) Build the small v3 corpus. Every row records its source so the mixture is
# auditable. Base replay is generated before training, using the zero-init LoRA.
from collections import Counter
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel

SYS_GENERAL = "You are a helpful, accurate assistant. Answer clearly and concisely."
SYS_THEORIA = (
    "You are Theoria, an offline mathematics and science assistant designed "
    "for budget laptops. Be accurate, concise, and honest about limitations."
)
SYS_MATH = (
    "You are Theoria. Solve the problem step by step and put only the final "
    "mathematical result in \\boxed{}."
)
SYS_LEAN = (
    "You are Theoria. Return valid Lean 4 using core syntax and tactics. "
    "Put code in one ```lean block; never invent Mathlib declarations."
)

def row(source, system, user, assistant):
    return {"source": source, "messages": [
        {"role": "system", "content": system.strip()},
        {"role": "user", "content": user.strip()},
        {"role": "assistant", "content": assistant.strip()},
    ]}

rows = []

# Identity/local behavior: two system contexts, not many copies of one answer.
identity = json.load(open(SEED_PATH))
for section in ("identity_qa", "chitchat", "boundaries", "shona"):
    for pair in identity.get(section, []):
        rows.append(row("identity", SYS_THEORIA, pair["q"], pair["a"]))
        rows.append(row("identity", SYS_GENERAL, pair["q"], pair["a"]))

# Clean GSM8K: remove calculator traces and the #### template that damaged prior tunes.
gsm = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=SEED)
for ex in gsm.select(range(MATH_N)):
    reasoning, final = ex["answer"].rsplit("####", 1)
    reasoning = re.sub(r"<<[^>]*>>", "", reasoning).strip()
    answer = f"{reasoning}\n\nTherefore, the final answer is \\boxed{{{final.strip()}}}."
    rows.append(row("math", SYS_MATH, ex["question"], answer))

# Short science only; reject diagram/figure questions and truncate source prose.
sciq = load_dataset("allenai/sciq", split="train").shuffle(seed=SEED)
kept = 0
for ex in sciq:
    blob = " ".join(str(ex.get(k, "")) for k in ("question", "support"))
    if re.search(r"\b(figure|diagram|image|shown above|shown below)\b", blob, re.I):
        continue
    support = (ex.get("support") or "").strip()
    sentences = re.split(r"(?<=[.!?])\s+", support)
    explanation = " ".join(sentences[:2]).strip()
    answer = ex["correct_answer"].strip()
    text = f"{explanation} Therefore, the answer is {answer}." if explanation else f"The answer is {answer}."
    rows.append(row("science", SYS_THEORIA, ex["question"], text))
    kept += 1
    if kept >= SCIENCE_N:
        break

# Lean: TheoriaKit core curriculum (verified offline) + deduped sorry_fill.
# Avoid lean_bridge.jsonl — much of it is uncompiled Mathlib/Lean3-style data.
# sorry_fill alone only has ~8 unique users after dedup.
THEORIAKIT_LEAN = [
    ("n + 0 = n", "add_zero' (n : Nat) : n + 0 = n := rfl", "Definitional: n + 0 is n."),
    ("0 + n = n", "zero_add' (n : Nat) : 0 + n = n := by simp", "`simp` unfolds 0 + n."),
    ("a + b = b + a", "add_comm' (a b : Nat) : a + b = b + a := Nat.add_comm a b", "Use Nat.add_comm."),
    ("(a + b) + c = a + (b + c)", "add_assoc' (a b c : Nat) : (a + b) + c = a + (b + c) := Nat.add_assoc a b c", "Nat.add_assoc."),
    ("a * b = b * a", "mul_comm' (a b : Nat) : a * b = b * a := Nat.mul_comm a b", "Nat.mul_comm."),
    ("n * 1 = n", "mul_one' (n : Nat) : n * 1 = n := Nat.mul_one n", "Nat.mul_one."),
    ("0 < n + 1", "succ_pos' (n : Nat) : 0 < n + 1 := by omega", "`omega` for Nat inequalities."),
    ("a ≤ a + b", "le_add_right' (a b : Nat) : a ≤ a + b := by omega", "Adding a Nat cannot decrease."),
    ("(2 * n) % 2 = 0", "two_mul_is_even (n : Nat) : (2 * n) % 2 = 0 := by omega", "Evenness via omega."),
    ("n * 0 = 0", "mul_zero' (n : Nat) : n * 0 = 0 := by simp", "`simp` for mul by zero."),
    ("0 * n = 0", "zero_mul' (n : Nat) : 0 * n = 0 := by simp", "`simp` for 0 * n."),
    ("n < n + 1", "lt_succ_self (n : Nat) : n < n + 1 := by omega", "Strict increase by 1."),
    ("¬ n < n", "not_lt_self (n : Nat) : ¬ n < n := by omega", "Irreflexivity of <."),
    ("n % 2 < 2", "mod_two_lt (n : Nat) : n % 2 < 2 := by omega", "Remainder bound."),
    ("n ^ 2 = n * n", "pow_two_eq_mul_self (n : Nat) : n ^ 2 = n * n := by\n  rw [Nat.pow_succ, Nat.pow_one]", "Expand power."),
    ("a + c = b + c → a = b", "add_right_cancel (a b c : Nat) (h : a + c = b + c) : a = b := by omega", "Cancel addend with omega."),
]
seen_lean = set()
for informal, formal, note in THEORIAKIT_LEAN:
    user = (
        "Fill the sorry in this Lean 4 theorem:\n"
        f"```lean\ntheorem {formal.split(':=')[0].strip()} := by sorry\n```"
    )
    assistant = f"{note}\n\n```lean\ntheorem {formal}\n```"
    key = hashlib.sha256(user.encode()).hexdigest()
    if key not in seen_lean:
        rows.append(row("lean", SYS_LEAN, user, assistant))
        seen_lean.add(key)
    # Also teach free-form formalization of the same fact.
    user2 = f"Write a Lean 4 theorem proving: {informal}. Use one ```lean block."
    assistant2 = f"{note}\n\n```lean\ntheorem {formal}\n```"
    key2 = hashlib.sha256(user2.encode()).hexdigest()
    if key2 not in seen_lean:
        rows.append(row("lean", SYS_LEAN, user2, assistant2))
        seen_lean.add(key2)

with open(SORRY_PATH) as f:
    for line in f:
        item = json.loads(line)
        user = next(m["content"] for m in item["messages"] if m["role"] == "user")
        assistant = next(m["content"] for m in item["messages"] if m["role"] == "assistant")
        key = hashlib.sha256(user.encode()).hexdigest()
        if key not in seen_lean:
            rows.append(row("lean", SYS_LEAN, user, assistant))
            seen_lean.add(key)

# Endogenous replay: ask the untouched base model to answer diverse general
# instructions, then preserve those answers during SFT.
dolly = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=SEED)
replay_prompts = []
for ex in dolly:
    if ex["category"] not in {"open_qa", "general_qa", "brainstorming", "classification"}:
        continue
    prompt = ex["instruction"].strip()
    if ex.get("context", "").strip():
        prompt += "\n\nContext:\n" + ex["context"].strip()
    if 20 <= len(prompt) <= 600:
        replay_prompts.append(prompt)
    if len(replay_prompts) >= REPLAY_N:
        break

# Silence Qwen generation_config.max_length=40960 vs max_new_tokens warning.
if getattr(model, "generation_config", None) is not None:
    model.generation_config.max_length = None

FastLanguageModel.for_inference(model)
for start in range(0, len(replay_prompts), 4):
    batch = replay_prompts[start:start + 4]
    texts = [tokenizer.apply_chat_template(
        [{"role": "system", "content": SYS_GENERAL}, {"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    ) for p in batch]
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                       max_length=MAX_SEQ_LEN).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=160, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    for i, output in enumerate(outputs):
        answer = tokenizer.decode(output[inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if 10 <= len(answer) <= 1200 and "<think>" not in answer:
            rows.append(row("replay", SYS_GENERAL, batch[i], answer))
    if start % 40 == 0:
        print(f"replay {min(start + 4, len(replay_prompts))}/{len(replay_prompts)}")
FastLanguageModel.for_training(model)

# Exact deduplication and artifact rejection.
unique = {}
for item in rows:
    packed = json.dumps(item["messages"], sort_keys=True, ensure_ascii=False)
    if any(bad in item["messages"][-1]["content"] for bad in ("####", "<<", "<think>")):
        continue
    unique[hashlib.sha256(packed.encode()).hexdigest()] = item
rows = list(unique.values())
random.Random(SEED).shuffle(rows)

counts = Counter(x["source"] for x in rows)
print("Corpus:", counts, "total=", len(rows))
assert len(rows) >= 450, "Corpus unexpectedly small"
assert counts["replay"] >= 80, "Too little base replay"
assert counts["lean"] >= 15, "Lean upload did not yield enough unique rows"
assert not any("####" in m["content"] or "<<" in m["content"] for r in rows for m in r["messages"])

raw_ds = Dataset.from_list(rows)
print(raw_ds[0])

In [ ]:
# 5) Freeze a small behavioral gate and a held-out GSM8K slice before training.
# The same prompts and decoding are used after training.
PROBES = {
    "identity": "What is your name, and where do you run?",
    "greeting": "hello",
    "quadratic": "Solve x^2 - 5x + 6 = 0. Show each step briefly.",
    "derivative": "Find the derivative of sin(x^2) with respect to x.",
    "science": "Why does ice float on liquid water? Answer in three sentences.",
    "counterexample": "Prove or disprove: for every integer n, n^2 > n.",
    "lean": "Write a Lean 4 theorem and proof that n + 0 = n for every natural n. Use one ```lean block.",
}

def generate_one(prompt, max_new_tokens=320):
    FastLanguageModel.for_inference(model)
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYS_THEORIA}, {"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def run_probes(label):
    answers = {name: generate_one(prompt) for name, prompt in PROBES.items()}
    print(f"\n===== {label} =====")
    for name, answer in answers.items():
        print(f"\n[{name}]\n{answer[:900]}")
    return answers

def probe_checks(a):
    normalized = {k: v.lower().replace(" ", "") for k, v in a.items()}
    return {
        "identity": "theoria" in normalized["identity"],
        "greeting": 2 <= len(a["greeting"].split()) <= 50,
        "quadratic": "2" in a["quadratic"] and "3" in a["quadratic"],
        "derivative": "cos" in normalized["derivative"] and "2x" in normalized["derivative"],
        "science": "lessdense" in normalized["science"],
        "counterexample": any(x in normalized["counterexample"] for x in ("false", "counterexample")),
        "lean": "```lean" in a["lean"] and any(x in a["lean"] for x in ("theorem", "example")),
        "clean": all(x not in "\n".join(a.values()) for x in ("####", "<<", "<think>")),
    }

def extract_number(text):
    boxed = re.findall(r"\\boxed\{([^}]*)\}", text)
    source = boxed[-1] if boxed else text
    nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", source)
    return nums[-1].replace(",", "").rstrip(".") if nums else None

EVAL_N = 40
gsm_test = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=SEED).select(range(EVAL_N))
def eval_gsm8k(label):
    correct = 0
    for i, ex in enumerate(gsm_test):
        gold = ex["answer"].rsplit("####", 1)[-1].strip().replace(",", "")
        pred = extract_number(generate_one(ex["question"], max_new_tokens=320))
        correct += int(pred == gold)
        if (i + 1) % 10 == 0: print(f"{label} GSM8K {i+1}/{EVAL_N}")
    acc = correct / EVAL_N
    print(f"[{label}] GSM8K: {correct}/{EVAL_N} = {acc:.1%}")
    return acc

base_answers = run_probes("BASE")
base_checks = probe_checks(base_answers)
base_acc = eval_gsm8k("base")
print("Base checks:", base_checks)
# Identity and Lean are adaptation targets, so the base may fail those. It must
# still pass the capabilities we refuse to damage.
for key in ("greeting", "quadratic", "derivative", "science", "counterexample", "clean"):
    assert base_checks[key], f"Base failed {key}; inspect the prompt before training"
FastLanguageModel.for_training(model)

In [ ]:
# 6) Apply Qwen3's non-thinking chat template, split deterministically, and train
# for exactly one epoch. Only assistant tokens contribute to the loss.
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False,
        enable_thinking=False,
    )}

formatted = raw_ds.map(to_text, remove_columns=raw_ds.column_names)
split = formatted.train_test_split(test_size=0.08, seed=SEED)
print(split)

args = SFTConfig(
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # effective batch 16
    num_train_epochs=1,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    optim="adamw_8bit",
    seed=SEED,
    data_seed=SEED,
    output_dir="theoria-v3-checkpoints",
    report_to="none",
)

# TRL changed `tokenizer` to `processing_class`; support both Colab stacks.
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer,
                         train_dataset=split["train"], eval_dataset=split["test"], args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                         train_dataset=split["train"], eval_dataset=split["test"], args=args)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Fail fast if a library/chat-template change caused every token to be masked.
sample_labels = trainer.train_dataset[0]["labels"]
supervised = sum(int(x != -100) for x in sample_labels)
assert supervised > 0, "Assistant-only masking matched zero tokens; stop and inspect the Qwen chat template"
print(f"Masking preflight: {supervised}/{len(sample_labels)} tokens supervised in sample 0")

stats = trainer.train()
print(stats)
print("Training finished. Do not export until cells 7–9 pass.")

In [ ]:
# 7) Run the exact same behavioral and held-out gates after training.
tuned_answers = run_probes("TUNED")
tuned_checks = probe_checks(tuned_answers)
tuned_acc = eval_gsm8k("tuned")

required = ["identity", "greeting", "quadratic", "derivative", "science",
            "counterexample", "lean", "clean"]
probe_pass = all(tuned_checks[k] for k in required)
math_pass = tuned_acc >= base_acc
SHIP_CANDIDATE = bool(probe_pass and math_pass)

print("\nBase checks: ", base_checks)
print("Tuned checks:", tuned_checks)
print(f"GSM8K base {base_acc:.1%} -> tuned {tuned_acc:.1%}")
print("SHIP_CANDIDATE =", SHIP_CANDIDATE)
if not SHIP_CANDIDATE:
    print("STOP: do not export or submit this adapter. Ship the official base Q4_K_M.")

In [ ]:
# 8) Save an audit record and the small LoRA adapter. Saving the adapter is
# useful even when the merged GGUF is rejected: it preserves the experiment.
from pathlib import Path

report = {
    "base_model": BASE_MODEL,
    "seed": SEED,
    "lora_r": LORA_R,
    "learning_rate": LEARNING_RATE,
    "rows": len(rows),
    "mixture": dict(counts),
    "base_checks": base_checks,
    "tuned_checks": tuned_checks,
    "base_gsm8k_40": base_acc,
    "tuned_gsm8k_40": tuned_acc,
    "ship_candidate": SHIP_CANDIDATE,
    "base_answers": base_answers,
    "tuned_answers": tuned_answers,
}
Path("theoria-v3-eval.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
model.save_pretrained("theoria-v3-lora")
tokenizer.save_pretrained("theoria-v3-lora")
print(json.dumps({k: report[k] for k in report if k not in {"base_answers", "tuned_answers"}}, indent=2))
print("\nManual check before export: no malformed LaTeX, no repeated tail, Lean code is plausible.")

In [ ]:
# 9) Merge and quantize only after the automatic gate passes.
# Q4_K_M is the target: on the 4-vCPU VM it was ~1.03 GiB and ~25 llama-bench t/s.
# Unsloth may write into theoria-v3-gguf/ OR theoria-v3-gguf_gguf/ — search both.
assert SHIP_CANDIDATE, "Gate failed: do not export this run"

import glob, shutil, time
from google.colab import drive

model.save_pretrained_gguf(
    "theoria-v3-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

candidates = (
    glob.glob("**/qwen3*.Q4_K_M.gguf", recursive=True)
    + glob.glob("**/*Q4_K_M*.gguf", recursive=True)
    + glob.glob("theoria-v3-gguf*/*.gguf")
)
# Prefer Q4_K_M over any leftover F16
candidates = sorted(
    {p for p in candidates if p.endswith(".gguf")},
    key=lambda p: (0 if "Q4_K_M" in p or "q4_k_m" in p.lower() else 1, -os.path.getsize(p)),
)
print("Found GGUFs:", candidates)
assert candidates, "Unsloth did not produce a GGUF under theoria-v3-gguf*"
target = "theoria-v3-q4_k_m.gguf"
shutil.copy2(candidates[0], target)  # copy so we keep Unsloth's original too
print(f"{target}: {os.path.getsize(target) / 1024**3:.2f} GiB  (from {candidates[0]})")
assert os.path.getsize(target) > 500_000_000, f"{target} looks truncated"

def mount_drive(retries=3):
    """Colab Drive mounts often flake once; retry before giving up."""
    last = None
    for i in range(1, retries + 1):
        try:
            drive.mount("/content/drive", force_remount=(i > 1))
            assert os.path.isdir("/content/drive/MyDrive"), "MyDrive missing after mount"
            return True
        except Exception as e:
            last = e
            print(f"Drive mount attempt {i}/{retries} failed: {e}")
            time.sleep(2 * i)
    print(
        "WARNING: Drive mount failed. GGUF is still on this runtime at:\n"
        f"  /content/{target}\n"
        "Do NOT Runtime → Restart. Re-run the next cell after fixing Drive, or run:\n"
        "  from google.colab import drive; drive.mount('/content/drive', force_remount=True)"
    )
    if last:
        print("Last error:", repr(last))
    return False

if mount_drive():
    out_dir = "/content/drive/MyDrive/Theoria-v3"
    os.makedirs(out_dir, exist_ok=True)
    dest = f"{out_dir}/{target}"
    shutil.copy2(target, dest)
    print("Copied GGUF to", dest, f"({os.path.getsize(dest) / 1024**3:.2f} GiB)")
else:
    print("Local GGUF OK — save to Drive from cell 10 once mount works.")


In [ ]:
# 10) Persist remaining artifacts. Safe to re-run if cell 9's Drive copy failed.
# Do NOT Runtime → Restart until the GGUF is on Drive (or you downloaded it).
from google.colab import drive, files
import time

target = "theoria-v3-q4_k_m.gguf"
assert os.path.isfile(target), (
    f"Missing {target}. If you still have Unsloth's file, run:\n"
    "  import glob, shutil\n"
    "  src = sorted(glob.glob('**/qwen3*.Q4_K_M.gguf', recursive=True))[0]\n"
    "  shutil.copy2(src, 'theoria-v3-q4_k_m.gguf')"
)

def mount_drive(retries=4):
    last = None
    for i in range(1, retries + 1):
        try:
            drive.mount("/content/drive", force_remount=(i > 1))
            assert os.path.isdir("/content/drive/MyDrive")
            return True
        except Exception as e:
            last = e
            print(f"Drive mount attempt {i}/{retries} failed: {e}")
            time.sleep(2 * i)
    raise RuntimeError(
        "Drive mount failed. Keep this runtime alive. In a new cell try:\n"
        "  from google.colab import drive\n"
        "  drive.mount('/content/drive', force_remount=True)\n"
        "Or download locally (slow):\n"
        "  from google.colab import files; files.download('theoria-v3-q4_k_m.gguf')\n"
        f"Last error: {last!r}"
    )

mount_drive()
out_dir = "/content/drive/MyDrive/Theoria-v3"
os.makedirs(out_dir, exist_ok=True)
shutil.copy2(target, f"{out_dir}/{target}")
if os.path.isfile("theoria-v3-eval.json"):
    shutil.copy2("theoria-v3-eval.json", out_dir)
if os.path.isdir("theoria-v3-lora"):
    shutil.make_archive("theoria-v3-lora", "zip", "theoria-v3-lora")
    shutil.copy2("theoria-v3-lora.zip", out_dir)
print("Saved to", out_dir)
print("GGUF bytes:", os.path.getsize(f"{out_dir}/{target}"))

if os.path.isfile("theoria-v3-eval.json"):
    files.download("theoria-v3-eval.json")

print("""
NEXT — do not update metadata yet:
1. Confirm MyDrive/Theoria-v3/theoria-v3-q4_k_m.gguf (~1.03 GiB).
2. Put it in model/candidates/theoria-v3-q4_k_m.gguf.
3. Run scripts/probe_quality.py against base Q4 and this GGUF.
4. Upload to the 8 GB VM and run adtc-profiler --skip-accuracy.
5. Adopt only if quality remains >= base and profiler TPS/RAM win.
""")
